## Step 8: Functional Annotation
**Input:** Predicted protein sequences from Step 7 (`braker.aa`)  
**Output:** InterProScan results in `10-functional-annotation/interproscan/output/`; 
eggNOG results in `10-functional-annotation/eggnog/output/`; 
PHI-base BLAST results in `phibase_analysis/`  
**Tools:** InterProScan v5.75-106.0, eggNOG-mapper v2.1.13 
(DIAMOND v2.1.12 + MMseqs2 v17), BLAST+ (BLASTp), PHI-base v4.19  
**Key parameters:** PHI-base filter: identity ≥40%, alignment ≥100 aa, 
E-value ≤1e-10  
**Key finding:** 93% of proteome annotated by InterProScan (577,904 domain hits); 
4,416 unique Pfam domains; 3,317 KEGG KO terms; 
2,673 high-confidence PHI-base homologs  
**Reference:** Materials & Methods Section 5 — Nebli et al. (2025)

In [ ]:
export SN=3RR
export NCPUS=128
export proteome="$PWD/09-annot/prot_only/braker.aa"

# Note: The path to the proteome file assumes it was generated by BRAKER in the previous chapter.

# Set up

In [ ]:
alias interproscan_app="apptainer run docker://interpro/interproscan:5.75-106.0"
alias eggnog-mapper="apptainer run docker://quay.io/biocontainers/eggnog-mapper:2.1.13--pyhdfd78af_0"

mkdir -p 10-functional-annotation/interproscan
curl -o 10-functional-annotation/interproscan/interproscan-data-5.75-106.0.tar.gz http://ftp.ebi.ac.uk/pub/software/unix/iprscan/5/5.75-106.0/alt/interproscan-data-5.75-106.0.tar.gz

tar -pxzf 10-functional-annotation/interproscan/interproscan-data-5.75-106.0.tar.gz -C 10-functional-annotation/interproscan

# Run InterProScan

In [ ]:
mkdir -p 10-functional-annotation/interproscan/{input,temp,output}
cp $proteome 10-functional-annotation/interproscan/input/
cp $genome 10-functional-annotation/interproscan/input/


# We need to bind the data, input, temp, and output directories to the container.
# Note that we are using the full path to the directories.
DATADIR=$PWD/10-functional-annotation/interproscan/interproscan-5.75-106.0/data
INPUTDIR=$PWD/10-functional-annotation/interproscan/input
TEMPDIR=$PWD/10-functional-annotation/interproscan/temp
OUTPUTDIR=$PWD/10-functional-annotation/interproscan/output

apptainer exec \
    --bind "$DATADIR":/opt/interproscan/data \
    --bind "$INPUTDIR":/input \
    --bind "$TEMPDIR":/temp \
    --bind "$OUTPUTDIR":/output \
    ~/interpro_tmp/interproscan_5.75-106.0.sif \
    /opt/interproscan/interproscan.sh \
    --input /input/braker.aa \
    --output-dir /output \
    --tempdir /temp \
    --cpu "$NCPUS" \
    --goterms \
    --pathways

# Run EggNog-mapper

In [ ]:
mkdir -p 10-functional-annotation/eggnog/output

DATADIR=/fpool/opt/db/eggnog-data
INPUTFILE=$proteome
OUTPUTDIR=$PWD/10-functional-annotation/eggnog/output
OUTPUT_PREFIX=${SN}_eggnog

apptainer run \
    --bind $DATADIR:/data \
    --bind $(dirname $INPUTFILE):/input \
    --bind $OUTPUTDIR:/output \
    docker://quay.io/biocontainers/eggnog-mapper:2.1.13--pyhdfd78af_0 \
    emapper.py \
    -i /input/$(basename $INPUTFILE) \
    -o $OUTPUT_PREFIX \
    --output_dir /output \
    --data_dir /data \
    --dmnd_db /data/fungi.dmnd \
    --cpu $NCPUS \
    --override

# Pathogen-Host Interactions

## CONFIGURATION

In [ ]:
export NCPUS=32
export WORKDIR="$PWD/phibase_analysis"
export QUERY="braker.aa"          # your predicted proteins FASTA
export EVALUE="1e-5"

In [ ]:
# PHI-base download + BLAST analysis pipeline
# Pathogen-Host Interactions database analysis

mkdir -p "$WORKDIR"
cd "$WORKDIR"

# DOWNLOAD PHI-base
wget -O PHI-base.fas.gz \
"https://www.phi-base.org/images/Links_to_other_sites/PHI-base_current.fas.gz"

gunzip PHI-base.fas.gz

# CREATE BLAST DATABASE

makeblastdb \
-in PHI-base.fas \
-dbtype prot \
-parse_seqids \
-out PHIbase_DB


# RUN BLASTP

blastp \
-query "$QUERY" \
-db PHIbase_DB \
-out phibase_blast.tsv \
-evalue "$EVALUE" \
-num_threads "$NCPUS" \
-max_target_seqs 5 \
-outfmt "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore stitle"

# -------------------------
# FILTER BEST HITS
# identity >= 40%
# coverage >= 100 aa
# -------------------------
awk '$3 >= 40 && $4 >= 100' \
phibase_blast.tsv > phibase_filtered.tsv

# -------------------------
# SUMMARY
# -------------------------
echo "PHI-base analysis completed"
echo "Results:"
echo "  Raw BLAST   : phibase_blast.tsv"
echo "  Filtered    : phibase_filtered.tsv"